# 09 — Retry Policies, Rate Limiting, and Idempotency

Reliable clients handle transient failures without amplifying outages or duplicating actions. Practice bounded retries, exponential backoff, rate limits, and idempotency keys using simulated services.

*Advanced · 90–150 min*

## You will learn
- Differentiate transient from permanent failures
- Implement bounded retries with injected sleep
- Use rate-limiting and concurrency controls
- Explain idempotency keys and retry safety

## Examples

### 1. Classify failures before retrying

A retry is useful only when another attempt might succeed. Syntax errors, validation errors, and authentication errors generally require fixes, while select timeouts and server failures may be transient. Always cap attempts.

In [ ]:
class TransientError(Exception):
    pass
class PermanentError(Exception):
    pass

def unreliable():
    unreliable.calls += 1
    if unreliable.calls < 3:
        raise TransientError("temporary")
    return "ok"
unreliable.calls = 0
print("Calls needed to succeed:", 3)

**Think:** Why would retrying a bad password repeatedly be harmful?

### 2. Bounded exponential backoff

Backoff spaces out retries to avoid overwhelming a failing service. Inject a sleep function to test without waiting. In production, use jitter and obey server Retry-After semantics when applicable.

In [ ]:
def retry(call, *, attempts=4, delay=0.1, sleep=lambda _: None):
    if attempts < 1:
        raise ValueError("attempts must be >= 1")
    for index in range(attempts):
        try:
            return call()
        except TransientError:
            if index + 1 == attempts:
                raise
            sleep(delay * (2 ** index))

unreliable.calls = 0
delays = []
print(retry(unreliable, sleep=delays.append), delays)
assert delays == [0.1, 0.2]

**Think:** How can exponential backoff still synchronize clients without jitter?

### 3. Rate limiting versus concurrency limiting

A semaphore limits simultaneous operations; it does not by itself enforce a requests-per-second rate. A token bucket or server-provided rate limit can bound throughput over time. The demo shows a concurrency cap offline.

In [ ]:
import asyncio

async def bounded(values, limit):
    semaphore = asyncio.Semaphore(limit)
    active, peak = 0, 0
    async def process(value):
        nonlocal active, peak
        async with semaphore:
            active += 1
            peak = max(peak, active)
            await asyncio.sleep(0)
            active -= 1
            return value * 2
    results = await asyncio.gather(*(process(v) for v in values))
    return results, peak

results, peak = asyncio.run(bounded(range(8), limit=3))
print(results, peak)
assert peak <= 3

**Think:** Why doesn't limiting simultaneous requests guarantee fewer than N requests per second?

### 4. Idempotent requests and deduplication

A duplicate payment operation is dangerous. An idempotency key lets a server associate retries with one logical action. Real systems must persist keys atomically with outcomes, or race conditions can still cause duplication.

In [ ]:
class FakePayments:
    def __init__(self):
        self.results = {}
        self.sequence = 0
    def charge(self, key, amount):
        if key in self.results:
            return self.results[key]
        if amount <= 0:
            raise ValueError("positive amount required")
        self.sequence += 1
        result = {"payment_id": self.sequence, "amount": amount}
        self.results[key] = result
        return result

payments = FakePayments()
first = payments.charge("order-100", 50)
second = payments.charge("order-100", 50)
print(first, second)
assert first == second

**Think:** Why must a real service use atomic storage for idempotency records?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/02_concurrency_resilience_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/09', {1: 'e0eca59983fe8b6f', 2: '73977cb50902cb75', 3: '9748878b93c02349', 4: '8873a8fbec594108'})

**Exercise 1.** Backoff with injected jitter: the delays are capped and reproducible.

In [ ]:
import random

def delays(attempts, base, cap, rng):
    return [min(cap, base * 2 ** n) * rng.uniform(0.5, 1.0) for n in range(___)]

a = delays(5, 0.1, 1.0, random.Random(7))
b = delays(5, 0.1, 1.0, random.Random(___))
check(1, (len(a), a == b, max(a) <= 1.0))

**Exercise 2.** A bounded async worker pool: report the peak number running at once.

In [ ]:
import asyncio

async def main(limit):
    semaphore = asyncio.Semaphore(___)
    active = peak = 0
    async def work(_):
        nonlocal active, peak
        async with semaphore:
            active += 1
            peak = max(peak, active)
            await asyncio.sleep(0.01)
            active -= 1
    await asyncio.gather(*(work(i) for i in range(10)))
    return peak

check(2, run(main(3)))

**Exercise 3.** A token bucket with an injectable clock.

In [ ]:
class Bucket:
    def __init__(self, capacity, rate, clock):
        self.capacity, self.rate, self.clock = capacity, rate, clock
        self.tokens, self.last = capacity, clock()
    def allow(self):
        now = self.clock()
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * ___)
        self.last = now
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

now = [0.0]
b = Bucket(2, 1, lambda: now[0])
out = [b.allow(), b.allow(), b.allow()]
now[0] = 1.0
out.append(b.allow())
check(3, out)

**Exercise 4.** Retrying a non-idempotent charge charges twice. An idempotency key fixes it.

In [ ]:
charges = []
seen = {}

def charge(amount, key=None):
    if key is not None and key in seen:
        return ___
    charges.append(amount)
    receipt = f"r{len(charges)}"
    if key is not None:
        seen[key] = receipt
    return receipt

charge(10); charge(10)                     # a retry without a key
first = charge(20, key="k1")
retry = charge(20, key="k1")               # a retry with a key
check(4, (charges, first == retry))

## Common mistakes

- Never retry forever, and don't retry every error class.
- A semaphore is not a per-second rate limiter.
- Idempotency must be enforced atomically across concurrent clients.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Which error is worth retrying?

   - **a)** 404 Not Found
   - **b)** 400 Bad Request
   - **c)** 503 Service Unavailable / timeout

**2.** Why add jitter to backoff?

   - **a)** So many clients don't retry at the same instant
   - **b)** To make retries slower
   - **c)** Randomness is required

**3.** An idempotency key ensures…

   - **a)** Encryption
   - **b)** A repeated request has the same effect as one
   - **c)** Faster requests

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/09', {1: 'a61182c344df', 2: 'c2a9f61f0fa0', 3: 'b9dadf04c824'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [10 — Profiling, Memory, and Honest Benchmarking](10_profiling_memory_benchmarks.ipynb)